# Lesson 23 — Generalized Disjunctive Programming

## Learning objectives

1. Write a GDP with disjunctions over algebraic constraints, Boolean indicators and logic between them.
2. Convert a GDP to a MINLP with the big-M, multiple big-M and convex-hull reformulations, and compare their relaxations.
3. Solve a GDP *without* reformulating it, with logic-based algorithms (LOA, GLOA, LBB, RIC, enumeration, LD-SDA), and say which of them certify a global optimum.
4. Use `discopt`'s native GDP frontend (`either_or`, `Disjunct` blocks, indicators, `gdp_method=...`, `gdp_options=...`) and check it against a hand reformulation.
5. Recognize when a user-supplied big-M costs you the certificate.

## 1. Disjunctive programming

A *disjunction* is an "or": at least one branch must hold. Standard form {cite:p}`Raman1994,Grossmann2013`:

$$
\bigvee_{k \in K} \begin{bmatrix} Y_k \\ A_k x \le b_k \\ f_k(x) \le 0 \end{bmatrix}
$$

with $Y_k$ a Boolean variable. Exactly one $Y_k$ is True. The constraints in branch $k$ are active iff $Y_k$ is True.

Logical constraints between $Y_k$'s can be expressed (e.g., $Y_1 \lor Y_2 \Rightarrow \neg Y_3$).

GDP is *the* natural framework for process synthesis problems with discrete unit-selection decisions {cite:p}`Grossmann2013,Trespalacios2015`.

In `discopt` each disjunct's Boolean $Y_k$ is available as an **indicator** (`d.indicators[k]`, a `BooleanVar`), so the logic layer can refer to it directly (§7).

In `discopt` each disjunct's Boolean $Y_k$ is available as an **indicator** (`d.indicators[k]`, a `BooleanVar`), so the logic layer can refer to it directly (§7).

## 2. Big-M reformulation

Convert each disjunctive constraint $A_k x \le b_k$ to

$$A_k x \le b_k + M_k (1 - y_k)$$

with binary $y_k$ corresponding to $Y_k$, and $\sum_k y_k = 1$. Plus indicators on $f_k(x) \le 0$.

**Pros:** simple, smaller model.
**Cons:** weak relaxation; tightness depends on $M_k$ {cite:p}`Sawaya2005`.

## 3. Convex-hull reformulation

For each $k$, introduce a copy $x_k$ of the variables. Let $x = \sum_k x_k$. Constrain $A_k x_k \le b_k y_k$ and $\sum_k y_k = 1$, $y_k \in [0, 1]$. The shorthand $x_k \in y_k X_k$ unfolds into the explicit **bound-scaling** constraints $\underline x \, y_k \le x_k \le \overline x \, y_k$, which force $x_k = 0$ whenever $y_k = 0$. The convex hull of the disjunction is then:

$$x = \sum_k x_k, \quad A_k x_k \le b_k y_k, \quad \underline x \, y_k \le x_k \le \overline x \, y_k, \quad \sum_k y_k = 1, \; y \ge 0.$$

For nonlinear convex $f_k$, the natural extension uses the **perspective function**: for a convex $f$ on a domain $D$, the perspective $\tilde f(x, y) = y f(x / y)$ for $y > 0$ (with $\tilde f(0, 0) = 0$ and $\tilde f(x, 0) = +\infty$ otherwise) is jointly convex in $(x, y)$ and provides the tightest convex underestimator of "$y = 1 \Rightarrow f(x) \le t$, $y = 0 \Rightarrow x = 0$" {cite:p}`Trespalacios2015`. We meet it again systematically in Lesson 29.

**Pros:** tightest possible LP relaxation.
**Cons:** $|K|$ copies of variables (model size grows). Worth it when big-M is loose.

## 4. Two ways to solve a GDP

`discopt` separates two questions that are easy to blur:

**Axis 1: reformulate, then solve the algebraic model.** `gdp_method` picks the reformulation, and the resulting MILP/MINLP goes to the ordinary solver routes (HiGHS, convex MIQP B&B, spatial B&B):

| `gdp_method` | What it writes |
|---|---|
| `"big-m"` (default) | §2, with each $M$ derived from interval/LP bounds on the row over the variable box. |
| `"mbigm"` | Multiple big-M: one $M$ per (row, other disjunct) pair, from an LP over that disjunct {cite:p}`Trespalacios2015`. |
| `"hull"` | §3: disaggregated copies and perspectives. Tightest relaxation, most variables. |
| `"binary_multiplication"` | Each row $g(x) \le 0$ of disjunct $k$ becomes $y_k\, g(x) \le 0$. Exact and needs no bound on $g$, but the products are nonconvex, so the spatial B&B takes over. |
| `"auto"`, `"simplex"` | A per-disjunction big-M/hull choice from a structural advisor; an exact continuous (CNF) lowering with no selector binaries. |

**Axis 2: solve the disjunctive form natively.** A *logic-based* algorithm never builds one monolithic MINLP. It works on configurations (one disjunct chosen per disjunction) and solves the reduced subproblem of each configuration it visits, using a master problem or a tree to choose which configuration to try next {cite:p}`Chen2022PyomoGDP,Grossmann2013`:

| `gdp_method` | Algorithm | Bound |
|---|---|---|
| `"loa"` | Logic-based outer approximation {cite:p}`Turkay1996`: a MILP master of OA cuts on the convex rows proposes a configuration; its subproblem gives an upper bound and new cuts. | Valid when the rows it linearizes are certified convex. |
| `"gloa"` | Global LOA {cite:p}`Lee2001`: the master is a convex *relaxation* of every row, so the bound needs no convexity; subproblems are solved globally. | Always valid. |
| `"lbb"` | Logic-based branch and bound {cite:p}`Lee2000`: branch on disjunctions; a node drops the nonlinear rows of undecided disjuncts. | Valid. |
| `"ric"` | Relaxation with integer cuts: master = linear rows + no-good cuts; global subproblems. | Valid with a linear objective; otherwise only once every configuration is excluded. |
| `"enumerate"` | Every configuration, globally, after logic propagation. Refuses above `max_configurations`. | Min over all configurations. |
| `"ldsda"` | Logic-based discrete-steepest descent {cite:p}`Ovalle2025`: a **local** neighbourhood search over "external variables" (which disjunct each disjunction picks). | **None**: never `optimal`. |

Options for the native methods go in `gdp_options`, a dict with Pyomo GDPopt's key names (`iterlim`, `init_algorithm`, `starting_point`, `direction_norm`, ...) plus `discopt`'s own (`subproblem="global"`/`"nlp"`/`"local"`, `subproblem_time_limit`, `max_configurations`). An unknown key, or any `gdp_options` with a reformulation, raises `ValueError`.

**Practical recipe.** Start with the default big-M. Switch to hull (or try `mbigm`) when the $M$'s are large and the solve spends its time branching on indicators, especially with few disjuncts per disjunction. Reach for a logic-based method when each configuration is easy to solve but the monolithic relaxation is weak or huge, which is typical of process synthesis with many units. Use `ldsda` for a fast, good point, and pass that point on as a warm start rather than reading it as a proof.

The reactor cell below builds big-M **by hand** so you can see every row; the cell after it states the same disjunction **natively** and lets `solve(gdp_method=...)` do the work {cite:p}`Sawaya2005,Trespalacios2015`.

In [ ]:
# Standard discopt course imports. The lessons target the real
# `discopt.modeling.core.Model` API: `m.continuous(name, shape=..., lb=..., ub=...)`,
# `m.binary(name, shape=...)`, `m.integer(name, shape=..., lb=..., ub=...)`,
# `m.subject_to(...)`, `m.minimize(...) / .maximize(...)`, `m.solve(...)`.
# Result attributes: `r.status`, `r.objective`, `r.gap`, `r.bound`,
# `r.wall_time`, `r.node_count`, and `r.value(var)` for variable arrays.
import numpy as np
import discopt as do
import discopt.modeling as dm


In [ ]:
# Process synthesis: pick reactor type (CSTR or PFR) for one stage.
# CSTR: cost 100 + 5*F   (F flow)
# PFR : cost 60  + 8*F + 0.5*F^2
# Production target F = 4. Minimize cost.
#
# Below we implement the problem manually with the big-M reformulation, so every
# row is visible. The next cell states the same disjunction natively.
import numpy as np, discopt as do
from discopt.modeling import Model

m = Model("reactor")
F = m.continuous("F", lb=0, ub=10)
y_cstr = m.binary("y_cstr"); y_pfr = m.binary("y_pfr")
cost = m.continuous("cost", lb=0, ub=1000)
M = 1000

m.subject_to(F == 4)
m.subject_to(y_cstr + y_pfr == 1)                              # exactly one
# CSTR active: cost ≥ 100 + 5F  (and ≤ same, when active)
m.subject_to(cost >= 100 + 5 * F - M * (1 - y_cstr))
m.subject_to(cost <= 100 + 5 * F + M * (1 - y_cstr))
# PFR active
m.subject_to(cost >= 60 + 8 * F + 0.5 * F**2 - M * (1 - y_pfr))
m.subject_to(cost <= 60 + 8 * F + 0.5 * F**2 + M * (1 - y_pfr))
m.minimize(cost)
r = m.solve()
print("chosen reactor:", "CSTR" if r.value(y_cstr) > 0.5 else "PFR")
print("cost          :", r.objective)


In [ ]:
# The same model, stated as a disjunction. Each inner list is a disjunct;
# exactly one is *selected* and its constraints enforced.
def reactor_gdp():
    m = dm.Model("reactor_gdp")
    F = m.continuous("F", lb=0, ub=10)
    cost = m.continuous("cost", lb=0, ub=1000)
    m.subject_to(F == 4)
    m.either_or([
        [cost == 100 + 5 * F],                  # CSTR
        [cost == 60 + 8 * F + 0.5 * F**2],      # PFR
    ], name="reactor")
    m.minimize(cost)
    return m

for method in ["big-m", "hull", "mbigm", "binary_multiplication", "loa", "enumerate"]:
    mg = reactor_gdp()
    rg = mg.solve(gdp_method=method)
    # selected_disjuncts reads the *source* rows at the point: which disjuncts hold.
    print(f"{method:22s} status={rg.status:8s} cost={rg.objective:.4f}  "
          f"holds={do.selected_disjuncts(mg, rg.x, tolerance=1e-4)}")   # {'reactor': [1]} -> PFR


`do.selected_disjuncts(m, r.x)` checks which disjuncts' rows **hold** at the returned point. It reads the source rows, not a selector, so the list can contain more than one index when disjuncts overlap: under select-one semantics a point may lie in a disjunct that was not selected (§7). For the selector itself, read the indicator (§7).

## 5. Reformulations against logic-based algorithms

A three-stage process chain: each stage picks one of three units. Unit $k$ at stage $s$ has a fixed cost, a linear operating cost and a yield $\eta$, so the stage-$s$ disjunction is

$$\bigvee_{k=1}^{3}\begin{bmatrix} Y_{sk} \\ c_s \ge \phi_{sk} + v_{sk} F_s \\ F_{s+1} = \eta_{sk} F_s \end{bmatrix},$$

and we buy feed $F_0$ at a convex cost $2F_0 + 0.05F_0^2$ to deliver at least 8 units. With $3^3 = 27$ configurations the true optimum is easy to check, which makes it a good bench for comparing the methods.

In [ ]:
import time

UNITS = [  # (fixed cost, cost per unit flow, yield) for 3 units at each of 3 stages
    [(10, 1.0, 0.90), (25, 0.4, 0.97), (18, 0.6, 0.93)],
    [(5, 2.0, 0.85), (20, 0.5, 0.95), (12, 1.1, 0.90)],
    [(8, 1.5, 0.80), (30, 0.2, 0.92), (15, 0.9, 0.86)],
]

def chain():
    m = dm.Model("chain")
    F = m.continuous("F", shape=(4,), lb=0, ub=20)    # flow into each stage + product
    c = m.continuous("c", shape=(3,), lb=0, ub=100)   # cost of each stage
    for s, units in enumerate(UNITS):
        m.either_or([[c[s] >= fc + vc * F[s], F[s + 1] == eta * F[s]]
                     for fc, vc, eta in units], name=f"stage{s}")
    m.subject_to(F[3] >= 8)
    m.minimize(dm.sum(c) + 2 * F[0] + 0.05 * F[0] ** 2)
    return m

methods = ["big-m", "mbigm", "hull", "binary_multiplication",          # reformulations
           "loa", "gloa", "lbb", "ric", "enumerate", "ldsda"]          # logic-based
print(f"{'method':22s} {'status':9s} {'objective':>10s} {'bound':>10s} {'root bound':>11s} "
      f"{'nodes':>5s} {'time':>6s}  units chosen")
results = {}
for meth in methods:
    m = chain()
    t0 = time.perf_counter()
    r = m.solve(gdp_method=meth, time_limit=60)
    dt = time.perf_counter() - t0
    results[meth] = r
    fmt = lambda v: f"{v:10.4f}" if v is not None else f"{'-':>10s}"
    sel = do.selected_disjuncts(m, r.x, tolerance=1e-6)
    print(f"{meth:22s} {r.status:9s} {fmt(r.objective)} {fmt(r.bound)} {fmt(r.root_bound):>11s} "
          f"{r.node_count:5d} {dt:6.2f}  {[sel[f'stage{s}'] for s in range(3)]}")


Read the table column by column.

- **Every method except `ldsda` certifies the same optimum, 96.83** (units 1, 2 and 3 at stages 0, 1 and 2), with a bound equal to the objective.
- **Root bound.** The LP/QP relaxation at the root of the reformulated model is $\approx 0$ for big-M and multiple big-M and about $79.9$ for the hull. Relaxing the binaries lets big-M switch every cost row off at once. The hull's disaggregated copies keep each $c_s$ tied to a convex combination of the three units' costs. This is §3's "tightest relaxation" in numbers. Here the tree is small, so the tighter root buys few nodes, and the bigger hull model is slower. On a larger model the root gap grows into an exponential difference in nodes.
- **`binary_multiplication`** has no $M$ at all, but its products $y_k\,g(x)$ are nonconvex, so it goes to the spatial B&B (`r.algorithm_route` says which route ran) with a root bound no better than big-M's.
- **Logic-based methods** report `node_count = 0` and `termination = None`, because they do not run a B&B tree of their own. Their progress is in `r.solver_stats` (next cell). They solve a few small subproblems instead of one large relaxation.
- **`ldsda` stops at a local optimum**, 100.19, a different unit at stage 2, with `status="feasible"` and `bound=None`. No neighbour that changes one stage's choice improves it, but the global optimum changes a stage *and* compensates elsewhere. A local method's answer is an incumbent, never a certificate. `discopt` reports it that way rather than calling it optimal.

In [ ]:
# Inside the logic-based methods.
r = chain().solve(gdp_method="loa")
st = r.solver_stats
print("LOA:", st["iterations"], "iterations,", st["n_subproblems"], "subproblems, stop:", st["stop_reason"])
for it, lb, ub, t in st["trace"]:                 # (iteration, lower bound, incumbent, time)
    print(f"  iter {it}: LB = {lb:9.4f}   UB = {ub:9.4f}   t = {t:.2f}s")

# LD-SDA is local: its answer depends on where it starts. starting_point gives,
# per disjunction, the 1-based index of the disjunct to start from.
for sp in [None, [2, 2, 2], [3, 3, 3]]:
    opts = {} if sp is None else {"starting_point": sp}
    r = chain().solve(gdp_method="ldsda", gdp_options=opts)
    print(f"ldsda from {str(sp or 'default'):10s}: {r.status}, objective {r.objective:.4f}")

# enumerate counts before it solves anything, and refuses rather than truncates.
try:
    chain().solve(gdp_method="enumerate", gdp_options={"max_configurations": 10})
except ValueError as e:
    print(str(e).split(";")[0])


LOA closes the gap in two iterations. The first master proposes a configuration whose subproblem gives the incumbent, and the OA cuts from it lift the master bound to meet that incumbent. LD-SDA's answer depends on where it starts: from all-second units it reaches the global optimum, and from all-third units it stops at yet another local optimum (99.19). That is typical of a local method, and it is why `ldsda` is better used as an incumbent generator than as a proof.

## 6. User-supplied big-M and the certificate

You can pass your own $M$: `either_or(..., big_m=50)` (one value) or a dict mapping a row to its $M$. A too-small $M$ cuts off feasible points and can make a wrong answer look optimal. `discopt` therefore checks every user $M$ when it lowers the model:

- **certified** if it is at least the interval or LP bound of the row over the region where the row is relaxed. The solve proceeds as usual.
- **refused** with an error if the LP finds a feasible point that the value would cut.
- **neither**: the value is used, but the certificate is withheld. `optimal` becomes `feasible`, `infeasible` becomes `unknown`, and `bound` is `None`.

A hull solve does not read `big_m`, so its certificate is unaffected.

In [ ]:
def reactor_with_M(big_m):
    m = dm.Model("reactor_M")
    F = m.continuous("F", lb=0, ub=10)
    cost = m.continuous("cost", lb=0, ub=1000)
    m.subject_to(F == 4)
    m.either_or([[cost == 100 + 5 * F], [cost == 60 + 8 * F + 0.5 * F**2]],
                name="reactor", big_m=big_m)
    m.minimize(cost)
    return m

for M in [None, 2000, 50, 5]:
    r = reactor_with_M(M).solve(gdp_method="big-m")
    print(f"big_m={str(M):5s} status={r.status:8s} objective={r.objective}  bound={r.bound}  "
          f"certified={r.gap_certified}")
print("hull ignores big_m:", reactor_with_M(5).solve(gdp_method="hull").status)


With `big_m=50` the answer happens to be right, but `discopt` cannot prove that $M = 50$ is large enough (the logged warning names the LP bound it would need), so it does not certify the result. With `big_m=5` the lowered model has no feasible point, and an infeasibility that rests on an uncertified $M$ is reported as `unknown`, not `infeasible`. In both cases the solver refuses to claim more than it can prove.

## 7. Indicators, logic, and "or" versus "exactly one"

Reading `d.indicators` on the object `either_or` returns gives one `BooleanVar` per disjunct. Use them in logic with `m.logical(...)` (`.implies`, `&`, `|`, `~`, `dm.lor`, ...) or in the cardinality helpers. After a solve, read the selector with `r.value(b.variable)`.

**Select-one, not exactly-one-true.** `either_or` *selects* exactly one disjunct and enforces it. An unselected disjunct's rows may still happen to hold. So `indicator = 0` means "not selected", **not** "this predicate is false". The stricter reading ("exactly one predicate holds") is a different semantics that `discopt` declares but does not implement, and asking for it raises an error rather than returning an approximation.

**At least one.** `semantics="or"` asks for *at least one* selected disjunct. Big-M (and `"auto"`, which routes OR there), `binary_multiplication` and the logic-based methods support it. `"hull"` and `"mbigm"` refuse it rather than silently solving the select-one problem. With one-way activation, both semantics project onto the same union of disjuncts in $x$. The difference appears when the indicators carry costs or appear in logic: under OR, a solution may select (and pay for) two disjuncts at once.

In [ ]:
# Indicators and logic: if region a picks its upper interval, region b must too.
m = dm.Model("indicators")
x = m.continuous("x", lb=0, ub=10)
y = m.continuous("y", lb=0, ub=10)
a = m.either_or([[x <= 3], [x >= 7]], name="a")
b = m.either_or([[y <= 2], [y >= 8]], name="b")
m.logical(a.indicators[1].implies(b.indicators[1]))
m.minimize((x - 8) ** 2 + (y - 1) ** 2)
r = m.solve(gdp_method="hull")
print(f"objective {r.objective:.3f} at x={r.value(x):.3f}, y={r.value(y):.3f}")
print("a selectors:", [float(r.value(ind.variable)) for ind in a.indicators],
      " b selectors:", [float(r.value(ind.variable)) for ind in b.indicators])
# Without the implication the optimum would be x=8, y=1 (objective 0). With it,
# x >= 7 forces y >= 8 (cost 49), so the solver picks x <= 3 instead (cost 25).

# At-least-one semantics: inspect site A (x >= 2) OR site B (x <= 5), or both.
def or_model():
    m = dm.Model("or_demo")
    x = m.continuous("x", lb=0, ub=10)
    m.either_or([[x >= 2], [x <= 5]], name="site", semantics="or")
    m.minimize((x - 3) ** 2)
    return m
for meth in ["big-m", "lbb", "hull"]:
    try:
        r = or_model().solve(gdp_method=meth)
        print(f"OR with {meth:6s}: {r.status}, objective {r.objective:.4f}")
    except NotImplementedError as e:
        print(f"OR with {meth:6s}: refused -", str(e).split(",")[0])


## 8. Nested disjunctions and `Disjunct` blocks

Multi-branch disjunctions (3 or more options) and *nested* disjunctions are common in process synthesis: an outer choice of technology family, and inside it a choice of vendor. There are two ways to write the nesting.

- **Inline:** build the inner disjunction with `m.disjunction(...)` (it is *not* added to the model) and put it inside a disjunct of the outer `m.either_or(...)`. Big-M, multiple big-M and **hull** all lower this. The hull nests the inner disaggregation inside the outer one, which is never weaker than big-M.
- **Blocks:** `d = m.make_disjunct("name")` creates a block with a named indicator (`d.indicator`). Add rows with `d.subject_to(...)`, block-local variables with `d.variable(...)`, an inner disjunction with `d.add_disjunction([...])`, and finally `m.add_disjunction([d1, d2])`. This is the form the **logic-based methods** need for nesting, because every inner disjunct must have its own indicator.

`discopt.gdp.gdp_tree(m)` gives a read-only view of the whole structure (disjunctions, disjuncts, indicators, logic), whichever way it was written.

Logical constraints between disjunctions (e.g., "if reactor type A is chosen at stage 1 then separator type X cannot be chosen at stage 2") are Boolean implications on the indicators, as in §7.

In [ ]:
from discopt.gdp import gdp_tree

# Family A (x >= 4) has two vendors; family B is a single option.
def nested_inline():
    m = dm.Model("nested_inline")
    x = m.continuous("x", lb=0, ub=10)
    c = m.continuous("c", lb=0, ub=100)
    vendor = m.disjunction([[c >= 10 + 2 * x], [c >= 15 + x]], name="vendor")
    m.either_or([[x >= 4, vendor], [c >= 30 + 0.5 * x]], name="family")
    m.subject_to(x >= 6)
    m.minimize(c)
    return m

for meth in ["big-m", "hull"]:
    r = nested_inline().solve(gdp_method=meth)
    print(f"inline nesting, {meth:5s}: {r.status}, c = {r.objective:.3f}")

def nested_blocks():
    m = dm.Model("nested_blocks")
    x = m.continuous("x", lb=0, ub=10)
    c = m.continuous("c", lb=0, ub=100)
    A, B = m.make_disjunct("famA"), m.make_disjunct("famB")
    A.subject_to(x >= 4)
    v1, v2 = m.make_disjunct("vendor1"), m.make_disjunct("vendor2")
    v1.subject_to(c >= 10 + 2 * x)
    v2.subject_to(c >= 15 + x)
    A.add_disjunction([v1, v2], name="vendor")
    B.subject_to(c >= 30 + 0.5 * x)
    m.add_disjunction([A, B], name="family")
    m.subject_to(x >= 6)
    m.minimize(c)
    return m, (A, B, v1, v2)

for meth in ["big-m", "hull", "loa", "enumerate"]:
    m, blocks = nested_blocks()
    r = m.solve(gdp_method=meth)
    on = [d.name for d in blocks if r.value(d.indicator.variable) > 0.5]
    print(f"blocks, {meth:9s}: {r.status}, c = {r.objective:.3f}, selected {on}")

def show(node, depth=0):                       # walk the normalized GDP tree
    print("  " * depth + f"disjunction {node.name!r} ({node.kind}, {len(node.disjuncts)} disjuncts)")
    for d in node.disjuncts:
        print("  " * depth + f"  - disjunct {d.name!r}: {len(d.rows)} row(s)")
        for child in d.children:
            show(child, depth + 2)

m, _ = nested_blocks()
for root in gdp_tree(m).disjunctions:
    show(root)


With $x \ge 6$, vendor 2 ($15 + x = 21$) beats vendor 1 ($10 + 2x = 22$) and family B ($30 + 0.5x = 33$), so every method returns $c = 21$.

## 9. Checklist

1. State the disjunction natively. Check the result against a hand reformulation once, then trust the lowering.
2. Compare `r.root_bound` across `big-m`, `mbigm` and `hull` before you choose. The root gap predicts the tree size.
3. For synthesis problems with many configurations that are each cheap to solve, try `loa` (convex rows) or `gloa`/`lbb` (nonconvex rows).
4. Treat `ldsda` and any `status="feasible"` result as an incumbent, not a proof.
5. Leave `big_m` unset unless you can justify the value. An unjustified value costs you the certificate.

## References
{cite:p}`Raman1994,Grossmann2013,Trespalacios2015,Sawaya2005,Turkay1996,Lee2000,Lee2001,Ovalle2025,Chen2022PyomoGDP`.